# 🏥 Machine Learning para Datos Médicos — Preprocesamiento de Datos
## Taller Pre-Congreso CNIB 2025

---

> **¿Por qué es tan importante el preprocesamiento?**
> En medicina, los datos rara vez vienen "limpios". Los expedientes clínicos tienen campos en blanco, unidades inconsistentes, y variables que mezclan texto con números. Si alimentamos esos datos crudos a un modelo de ML, el resultado será basura — *garbage in, garbage out*. Por eso, antes de entrenar cualquier modelo hay que transformar los datos hasta dejarlos en un formato que los algoritmos puedan aprovechar.

---

### Librerías que usaremos
| Librería | Para qué sirve |
|---|---|
| `pandas` | Manipular tablas (DataFrames) |
| `numpy` | Operaciones numéricas vectorizadas |
| `matplotlib` / `seaborn` | Visualización de datos |
| `sklearn` | Transformaciones y modelos de ML |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 📑 Dataset: Wisconsin Breast Cancer (Cáncer de Mama)

**Contexto médico:** Este dataset proviene del Hospital de la Universidad de Wisconsin y contiene mediciones obtenidas de biopsias de tumor de mama mediante aspiración por aguja fina (FNA, *Fine Needle Aspiration*). Se miden características geométricas de los núcleos celulares visibles en la imagen: radio, textura, perímetro, área, suavidad, compacidad, concavidad, puntos cóncavos, simetría y dimensión fractal.

**Dato curioso 🔬:** El cáncer de mama es el más común en mujeres a nivel mundial. En México, es la primera causa de muerte por cáncer en mujeres. La biopsia FNA es mínimamente invasiva — se inserta una aguja delgada para aspirar células — y su análisis computacional puede igualar o superar la precisión del diagnóstico manual.

**Clases:**
- `0` → Maligno (canceroso)
- `1` → Benigno (no canceroso)

El dataset tiene **569 muestras** y **30 características numéricas**.


In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd
import numpy as np

# Cargamos el dataset clásico incluido en scikit-learn
data = load_breast_cancer()
cancer_df = pd.DataFrame(data.data, columns=data.feature_names)

# Visualizamos las primeras filas
cancer_df

In [ ]:
# Ver el nombre de todas las columnas (características)
# Cada columna representa una medición del núcleo celular
# Hay 3 grupos: mean (promedio), error, y worst (peor valor de la imagen)
cancer_df.columns

In [ ]:
# Resumen técnico del DataFrame:
# - dtype de cada columna (todos float64 en este caso)
# - si hay valores nulos (non-null count)
# - memoria usada
cancer_df.info()

In [ ]:
# Estadísticas descriptivas: media, desviación estándar, percentiles
# Muy útil para detectar escalas muy distintas entre variables
# (p. ej. 'mean area' puede llegar a 2500, mientras 'mean symmetry' ronda 0.2)
cancer_df.describe()

---

## 🧹 Limpieza de Datos (Data Cleaning)

**¿Qué son los valores nulos?** Son celdas vacías en la tabla — datos que simplemente no se registraron. En medicina esto es frecuente: el paciente no se midió el peso ese día, la glucosa no se tomó, la columna estaba en blanco en el expediente.

**Estrategias para tratar valores nulos:**
1. **Eliminación**: borrar las filas/columnas con nulos. Sencillo pero pierde información.
2. **Imputación por media/mediana/moda**: rellenar con el valor estadístico representativo. Es lo que usaremos.
3. **Imputación por modelo**: usar un modelo de ML para predecir el valor faltante. Más sofisticado.

> **Nota técnica 💡:** En este dataset original no hay nulos, así que los *simulamos* artificialmente para practicar la técnica.


In [ ]:
# Verificamos cuántos valores nulos hay en cada columna
# En el dataset original (limpio) debe ser 0 en todo
print("Valores nulos:\n", cancer_df.isnull().sum())

In [ ]:
# Simulamos valores nulos para practicar la técnica de imputación
# Insertamos 300 NaN en posiciones aleatorias del DataFrame
# (En la realidad, estos nulos ya vendrían en los datos crudos)
for i in range(300):
    cancer_df.iloc[np.random.randint(0, len(cancer_df)),
                   np.random.randint(0, len(cancer_df.columns))] = np.nan

print(cancer_df.isnull().sum())

In [ ]:
# IMPUTACIÓN POR MEDIA
# Para cada columna numérica, reemplazamos los NaN con el promedio de esa columna
# Esto preserva la distribución general sin eliminar filas

num_cols = cancer_df.select_dtypes(include=[np.number]).columns

cancer_df_clean = cancer_df.copy()   # Trabajamos sobre una copia para no perder el original
cancer_df_clean[num_cols] = cancer_df_clean[num_cols].fillna(cancer_df_clean[num_cols].mean())

print("Después de limpiar:\n", cancer_df_clean.isnull().sum())

In [ ]:
# Visualizamos el efecto de la imputación en la distribución de 'perimeter error'
# La distribución debería mantenerse similar — si cambia mucho, la imputación fue agresiva

fig, ax = plt.subplots(1, 2, figsize=(10,4))
sns.histplot(cancer_df['perimeter error'], kde=True, ax=ax[0], color='red')
ax[0].set_title('Perimeter error CON valores nulos')

sns.histplot(cancer_df_clean['perimeter error'], kde=True, ax=ax[1], color='green')
ax[1].set_title('Perimeter error DESPUÉS de imputación')

plt.tight_layout()
plt.show()

---

## ⚖️ Escalado de Características (Feature Scaling)

**¿Por qué escalar?** Los algoritmos basados en distancias (KNN, SVM) o en gradiente (redes neuronales) son sensibles a la magnitud de las variables. Si `mean area` va de 0 a 2500 y `mean symmetry` va de 0.1 a 0.3, el modelo "pensará" que el área es mucho más importante solo por su escala numérica.

**Dos técnicas comunes:**

| Técnica | Fórmula | Resultado | Cuándo usarla |
|---|---|---|---|
| **StandardScaler** | `(x - μ) / σ` | Media=0, Std=1 | Algoritmos que asumen distribución normal |
| **MinMaxScaler** | `(x - min) / (max - min)` | Rango [0, 1] | Redes neuronales, cuando se quiere rango fijo |

> **Dato curioso 🧮:** En 1957, cuando Frank Rosenblatt inventó el Perceptrón (precursor de las redes neuronales), el escalado ya era una preocupación. Sin embargo, no fue hasta los años 80-90 que se estandarizó como buena práctica obligatoria.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler: transforma para que la distribución tenga media=0 y varianza=1
scaler_std = StandardScaler()
cancer_df_scaled = cancer_df_clean.copy()
cancer_df_scaled[num_cols] = scaler_std.fit_transform(cancer_df_clean[num_cols])

# MinMaxScaler: comprime todos los valores al rango [0, 1]
scaler_mm = MinMaxScaler()
cancer_df_mm = cancer_df_clean.copy()
cancer_df_mm[num_cols] = scaler_mm.fit_transform(cancer_df_clean[num_cols])

# Visualizamos cómo cambian las posiciones relativas de los puntos
plt.figure(figsize=(6,6))
plt.scatter(cancer_df_clean['mean area'],   cancer_df_clean['worst radius'],   alpha=0.5, label='Original')
plt.scatter(cancer_df_scaled['mean area'],  cancer_df_scaled['worst radius'],  alpha=0.5, label='StandardScaler')
plt.scatter(cancer_df_mm['mean area'],      cancer_df_mm['worst radius'],      alpha=0.5, label='MinMaxScaler')
plt.legend()
plt.xlabel('mean area')
plt.ylabel('worst radius')
plt.title('Efecto del escalado sobre los datos')
plt.grid(ls='--', alpha=0.5)
plt.show()

---

## 🔎 Selección de Características (Feature Selection)

**¿Por qué seleccionar?** Con 30 variables, algunos modelos pueden sufrir la "maldición de la dimensionalidad" — demasiadas variables hacen que el espacio de búsqueda sea enorme y el modelo se sobreajuste (*overfitting*).

**ANOVA F-score:** Mide la varianza *entre* grupos dividida entre la varianza *dentro* del grupo. Una puntuación alta indica que esa variable separa bien las clases (benigno vs. maligno). `SelectKBest` selecciona las `k` variables con mayor puntuación.

> **Contexto médico 🩺:** En la práctica clínica, el radiólogo no mide las 30 características — se enfoca en las más diagnósticas. La selección automática de características imita ese razonamiento experto, pero de forma cuantitativa y reproducible.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Seleccionamos las 5 características más relevantes para distinguir benigno/maligno
selector = SelectKBest(f_classif, k=5)
selector.fit(cancer_df_scaled, data.target)

# Graficamos los scores de cada característica
scores = selector.scores_
plt.figure(figsize=(10,4))
sns.barplot(x=data.feature_names, y=scores,
            order=data.feature_names[np.argsort(scores)])
plt.xticks(rotation=90)
plt.title("Relevancia de características (ANOVA F-score)")
plt.ylabel("Puntuación F")
plt.tight_layout()
plt.show()

In [ ]:
# Nos quedamos solo con las 5 características seleccionadas
# Esto reduce la dimensionalidad de 30 → 5
cancer_df_top5 = cancer_df_scaled.iloc[:, selector.get_support()]

print("Forma original:", cancer_df_scaled.shape)
print("Después de selección:", cancer_df_top5.shape)
print("Columnas seleccionadas:", cancer_df_top5.columns.tolist())

---

## 📉 Extracción de Características (Feature Extraction)

**PCA (Análisis de Componentes Principales):** A diferencia de la selección (que *elige* columnas existentes), la extracción *crea* nuevas variables combinando las originales. PCA encuentra las direcciones de mayor varianza en los datos y proyecta todo sobre esas direcciones.

**Ventaja médica:** Permite visualizar datos de alta dimensión en 2D o 3D — algo imposible de hacer directamente con 30 variables. Si benigno y maligno se separan visualmente en el gráfico PCA, el modelo de clasificación tendrá trabajo más fácil.

> **Dato histórico 📚:** PCA fue desarrollado por Karl Pearson en 1901 (¡hace más de 120 años!), pero su aplicación en medicina computacional se disparó a partir de los años 90 con el análisis de microarreglos de ADN.


In [ ]:
from sklearn.decomposition import PCA

# Reducimos a 2 dimensiones para poder visualizar
pca = PCA(n_components=2)
cancer_df_PCA = pca.fit_transform(pd.get_dummies(cancer_df_top5))

plt.figure(figsize=(6,6))
plt.scatter(cancer_df_PCA[:,0], cancer_df_PCA[:,1],
            c=data.target, cmap='coolwarm', alpha=0.6)
plt.title('PCA con 2 componentes')
plt.xlabel('PC1 (mayor varianza)')
plt.ylabel('PC2 (segunda mayor varianza)')
plt.colorbar(label='Diagnóstico (0=Maligno, 1=Benigno)')
plt.grid(ls='--', alpha=0.5)
plt.show()

# Varianza explicada por cada componente
print(f"Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}")

---

## 🌀 Reducción de Dimensionalidad — t-SNE

**t-SNE** (*t-distributed Stochastic Neighbor Embedding*) es una técnica no lineal de reducción de dimensionalidad. A diferencia de PCA (que es lineal), t-SNE es excelente para revelar estructuras de cluster en los datos.

**¿Cuándo usar t-SNE vs PCA?**
- **PCA**: rápido, reproducible, útil como preprocesamiento o cuando la linealidad es suficiente.
- **t-SNE**: lento, no reproducible perfectamente, pero revela estructuras no lineales complejas.

> **⚠️ Cuidado:** t-SNE es solo para *visualización*. Los ejes no tienen interpretación directa (no son como las componentes de PCA). Nunca uses t-SNE para entrenar un modelo.


In [ ]:
from sklearn.manifold import TSNE

# Reducción a 2 dimensiones con t-SNE
# perplexity ~ número de vecinos cercanos que considera cada punto (suele ser 5-50)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
cancer_df_tsne = tsne.fit_transform(pd.get_dummies(cancer_df_top5))

plt.figure(figsize=(6,6))
plt.scatter(cancer_df_tsne[:,0], cancer_df_tsne[:,1],
            c=data.target, cmap='coolwarm', alpha=0.6)
plt.title('t-SNE con 2 dimensiones')
plt.xlabel('Dimensión 1')
plt.ylabel('Dimensión 2')
plt.colorbar(label='Diagnóstico (0=Maligno, 1=Benigno)')
plt.grid(ls='--', alpha=0.5)
plt.show()

---

## ✂️ División del Dataset (Train / Validation / Test Split)

**¿Por qué dividir los datos?**
- **Entrenamiento (Train, ~64%):** el modelo aprende los patrones.
- **Validación (Val, ~16%):** ajustamos los hiperparámetros del modelo sin "contaminar" el test.
- **Prueba (Test, ~20%):** evaluación final e imparcial del modelo.

> **Analogía médica 🩺:** Imagina que estás aprendiendo a diagnosticar EKGs. El conjunto de entrenamiento son los casos que estudias en el libro. La validación son los exámenes parciales donde corriges tu criterio. El test es el examen final con casos que nunca habías visto.

**`stratify=y`:** Asegura que la proporción de cada clase (benigno/maligno) sea igual en los tres conjuntos. Sin esto, podría quedar el test con puras muestras benignas y dar accuracy artificialmente alto.


In [ ]:
from sklearn.model_selection import train_test_split

# Primera división: 80% para train+val, 20% para test
X_train, X_test, y_train, y_test = train_test_split(
    cancer_df_clean, data.target,
    test_size=0.2,
    stratify=data.target,   # Mantener proporción de clases
    random_state=1           # Reproducibilidad
)

# Segunda división: del 80% restante, 20% para validación
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=1
)

# Visualizamos la distribución de los splits
labels = ['Train', 'Validation', 'Test']
sizes  = [len(y_train), len(y_val), len(y_test)]
plt.figure(figsize=(5,5))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90,
        colors=['#4c72b0','#dd8452', '#59a14f'])
plt.title('División del Dataset – Cáncer de Mama')
plt.show()

In [ ]:
# Verificamos las dimensiones de cada conjunto
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

---

## 📚 Para aprender más
- [Documentación de transformaciones en Scikit-Learn](https://scikit-learn.org/stable/data_transforms.html)
- Libro recomendado: *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* — Aurélien Géron
